# ЛР-02: Запчасти на ремонтные базы

## Student notebook: military 02

Этот notebook предназначен для самостоятельного решения.

Готового оптимального плана и заполненного решателя здесь нет.

## 1. Зачем нужен этот кейс

На ремонтные базы нужно доставить больше комплектов, чем сейчас есть на складах.

После этого notebook-а студент должен уметь:

1. замечать дефицит снабжения до запуска solver-а;
2. вводить фиктивного поставщика для балансировки;
3. фиксировать смысл неудовлетворённого спроса в выводе;
4. аккуратно интерпретировать полученную матрицу перевозок.

## 2. Исходные данные

Тип задачи: **открытая, спрос больше, чем запасов**.

### Запасы

| Поставщик | Объём |
| --- | --- |
| Склад A | 25 |
| Склад B | 30 |
| Склад C | 20 |

### Спрос

| Потребитель | Объём |
| --- | --- |
| Рембаза 1 | 15 |
| Рембаза 2 | 20 |
| Рембаза 3 | 18 |
| Рембаза 4 | 30 |

### Матрица затрат

| Откуда / Куда | Рембаза 1 | Рембаза 2 | Рембаза 3 | Рембаза 4 |
| --- | --- | --- | --- | --- |
| Склад A | 7 | 5 | 9 | 11 |
| Склад B | 6 | 4 | 7 | 8 |
| Склад C | 8 | 6 | 5 | 7 |

## 3. Что нужно сделать

Идите тем же маршрутом, что и в теории ЛР-02:

1. Проверьте баланс запасов и спроса.
2. Если задача открытая, добавьте фиктивный узел и поясните его смысл.
3. Запишите математическую модель через переменные $x_{ij}$.
4. Сформируйте `c`, `A_eq`, `b_eq`, `bounds` для `linprog`.
5. Проверьте, что `A_eq x = b_eq` действительно задаёт строки и столбцы.
6. После собственной попытки решите задачу через Python.
7. Кратко объясните реальные и фиктивные маршруты.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import linprog


DUMMY_SUPPLIER_NAME = "Фиктивный поставщик"
DUMMY_CONSUMER_NAME = "Фиктивный потребитель"
BALANCE_TOLERANCE = 1e-9

# Шаг 1: записываем данные задачи, не меняя числовую постановку.
supplier_names = [
    'Склад A',
    'Склад B',
    'Склад C',
]
consumer_names = [
    'Рембаза 1',
    'Рембаза 2',
    'Рембаза 3',
    'Рембаза 4',
]

supplies = np.array(
    [
        25,
        30,
        20,
    ],
    dtype=float,
)

demands = np.array(
    [
        15,
        20,
        18,
        30,
    ],
    dtype=float,
)

costs = np.array(
    [
        [7, 5, 9, 11],
        [6, 4, 7, 8],
        [8, 6, 5, 7],
    ],
    dtype=float,
)

# Шаг 2: выводим векторы и матрицу затрат в читаемом виде.
supply_df = pd.DataFrame({"запас": supplies}, index=supplier_names)
demand_df = pd.DataFrame({"спрос": demands}, index=consumer_names)
cost_df = pd.DataFrame(costs, index=supplier_names, columns=consumer_names)

print("Запасы поставщиков a_i:")
display(supply_df)

print("Спрос потребителей b_j:")
display(demand_df)

print("Матрица затрат c_ij:")
display(cost_df)

print("sum supply =", supplies.sum())
print("sum demand =", demands.sum())


Запасы поставщиков a_i:


,запас
Склад A,25.0
Склад B,30.0
Склад C,20.0


Спрос потребителей b_j:


,спрос
Рембаза 1,15.0
Рембаза 2,20.0
Рембаза 3,18.0
Рембаза 4,30.0


Матрица затрат c_ij:


,Рембаза 1,Рембаза 2,Рембаза 3,Рембаза 4
Склад A,7.0,5.0,9.0,11.0
Склад B,6.0,4.0,7.0,8.0
Склад C,8.0,6.0,5.0,7.0


sum supply = 75.0
sum demand = 83.0


## 4. Шаблон для самостоятельной сборки модели

Ниже намеренно оставлены `TODO`. Это не готовое решение, а guided skeleton:
он показывает правильные имена, порядок шагов и форму LP-модели, но ключевые
строки вы заполняете самостоятельно.


In [2]:
def balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
):
    """Готовит закрытую транспортную модель через фиктивный узел."""

    BALANCE_TOLERANCE = 1e-9

    balanced_supplies = supplies.astype(float).copy()
    balanced_demands = demands.astype(float).copy()
    balanced_costs = costs.astype(float).copy()
    balanced_supplier_names = list(supplier_names)
    balanced_consumer_names = list(consumer_names)

    balance_difference = balanced_supplies.sum() - balanced_demands.sum()

    # Добавляем фиктивного потребителя, если запасов больше
    if balance_difference > BALANCE_TOLERANCE:
        # Добавляем фиктивного потребителя
        balanced_demands = np.append(balanced_demands, balance_difference)
        balanced_consumer_names.append(DUMMY_CONSUMER_NAME)

        # Добавляем столбец с нулевыми затратами для фиктивного потребителя
        dummy_column = np.full((balanced_costs.shape[0], 1), dummy_cost)
        balanced_costs = np.column_stack([balanced_costs, dummy_column])

        balance_note = f"Задача открытая: запасов больше на {balance_difference:.1f}. Добавлен фиктивный потребитель."

    # Добавляем фиктивного поставщика, если спроса больше
    elif balance_difference < -BALANCE_TOLERANCE:
        # Добавляем фиктивного поставщика
        balanced_supplies = np.append(balanced_supplies, -balance_difference)
        balanced_supplier_names.append(DUMMY_SUPPLIER_NAME)

        # Добавляем строку с нулевыми затратами для фиктивного поставщика
        dummy_row = np.full((1, balanced_costs.shape[1]), dummy_cost)
        balanced_costs = np.vstack([balanced_costs, dummy_row])

        balance_note = f"Задача открытая: спроса больше на {-balance_difference:.1f}. Добавлен фиктивный поставщик."

    else:
        balance_note = "Задача закрытая: сумма запасов = сумме спроса"

    return (
        balanced_supplies,
        balanced_demands,
        balanced_costs,
        balanced_supplier_names,
        balanced_consumer_names,
        balance_note,
    )

In [3]:
def build_transport_lp(supplies, demands, costs):
    """Собирает c, A_eq, b_eq и bounds для linprog."""

    supplier_count, consumer_count = costs.shape
    variable_count = supplier_count * consumer_count

    # Вектор цели (стоимости перевозок)
    c = costs.flatten()

    # Матрица ограничений A_eq
    A_eq = np.zeros((supplier_count + consumer_count, variable_count))

    # Заполняем строки поставщиков
    for i in range(supplier_count):
        for j in range(consumer_count):
            idx = i * consumer_count + j
            A_eq[i, idx] = 1

    # Заполняем строки потребителей
    for j in range(consumer_count):
        for i in range(supplier_count):
            idx = i * consumer_count + j
            A_eq[supplier_count + j, idx] = 1

    # Вектор правых частей
    b_eq = np.concatenate([supplies, demands])

    # Границы переменных
    bounds = [(0.0, None) for _ in range(variable_count)]

    return {
        "c": c,
        "A_eq": A_eq,
        "b_eq": b_eq,
        "bounds": bounds,
        "route_index_hint": "supplier_idx * consumer_count + consumer_idx",
        "variable_count": variable_count,
    }

In [4]:
# Шаг 1: сбалансируйте задачу
(
    balanced_supplies,
    balanced_demands,
    balanced_costs,
    balanced_supplier_names,
    balanced_consumer_names,
    balance_note,
) = balance_transport_problem(
    supplies,
    demands,
    costs,
    supplier_names,
    consumer_names,
    dummy_cost=0.0,
)

print(balance_note)
print("balanced supply =", balanced_supplies.sum())
print("balanced demand =", balanced_demands.sum())
print("\nСбалансированные данные:")
print("Запасы:", balanced_supplies)
print("Спрос:", balanced_demands)

# Проверка баланса
assert np.allclose(balanced_supplies.sum(), balanced_demands.sum())

# Шаг 2: соберите LP-модель
lp_model = build_transport_lp(balanced_supplies, balanced_demands, balanced_costs)

# Вывод переменных и их стоимости
route_labels = [
    f"x_{supplier_idx + 1},{consumer_idx + 1}"
    for supplier_idx in range(len(balanced_supplier_names))
    for consumer_idx in range(len(balanced_consumer_names))
]

print("\nМатрица затрат (с фиктивным поставщиком):")
balanced_cost_df = pd.DataFrame(
    balanced_costs,
    index=balanced_supplier_names,
    columns=balanced_consumer_names
)
display(balanced_cost_df)

print("\nПеременные и их стоимости:")
display(pd.DataFrame({"переменная": route_labels, "стоимость c": lp_model["c"]}))

print("\nМатрица ограничений A_eq (первые 5 строк):")
display(pd.DataFrame(lp_model["A_eq"][:5], columns=route_labels))

print("\nПравые части ограничений b_eq:")
display(pd.DataFrame({"b_eq": lp_model["b_eq"]}))

Задача открытая: спроса больше на 8.0. Добавлен фиктивный поставщик.
balanced supply = 83.0
balanced demand = 83.0

Сбалансированные данные:
Запасы: [25. 30. 20.  8.]
Спрос: [15. 20. 18. 30.]

Матрица затрат (с фиктивным поставщиком):


,Рембаза 1,Рембаза 2,Рембаза 3,Рембаза 4
Склад A,7.0,5.0,9.0,11.0
Склад B,6.0,4.0,7.0,8.0
Склад C,8.0,6.0,5.0,7.0
Фиктивный поставщик,0.0,0.0,0.0,0.0



Переменные и их стоимости:


,переменная,стоимость c
0,"x_1,1",7.0
1,"x_1,2",5.0
2,"x_1,3",9.0
3,"x_1,4",11.0
4,"x_2,1",6.0
5,"x_2,2",4.0
6,"x_2,3",7.0
7,"x_2,4",8.0
8,"x_3,1",8.0
9,"x_3,2",6.0



Матрица ограничений A_eq (первые 5 строк):


,"x_1,1","x_1,2","x_1,3","x_1,4","x_2,1","x_2,2","x_2,3","x_2,4","x_3,1","x_3,2","x_3,3","x_3,4","x_4,1","x_4,2","x_4,3","x_4,4"
0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0



Правые части ограничений b_eq:


,b_eq
0,25.0
1,30.0
2,20.0
3,8.0
4,15.0
5,20.0
6,18.0
7,30.0


In [5]:
# Решение задачи
result = linprog(
    lp_model["c"],
    A_eq=lp_model["A_eq"],
    b_eq=lp_model["b_eq"],
    bounds=lp_model["bounds"],
    method="highs",
)

print(f"Успех решения: {result.success}")
print(f"Сообщение: {result.message}")

if result.success:
    # Преобразуем решение в матрицу плана перевозок
    plan = result.x.reshape(len(balanced_supplier_names), len(balanced_consumer_names))
    plan_df = pd.DataFrame(
        plan,
        index=balanced_supplier_names,
        columns=balanced_consumer_names,
        dtype=float
    )

    print("\nОптимальный план перевозок:")
    display(plan_df.round(2))

    # Добавим итоговые строки и столбцы
    plan_with_totals = plan_df.round(2).copy()
    plan_with_totals.loc['Итого'] = plan_with_totals.sum(axis=0)
    plan_with_totals['Итого'] = plan_with_totals.sum(axis=1)

    print("\nПлан перевозок с итогами:")
    display(plan_with_totals)

    # Проверка сумм по строкам и столбцам
    print("\nПроверка ограничений:")
    print("Строки (поставщики):")
    for i, name in enumerate(balanced_supplier_names):
        row_sum = plan[i, :].sum()
        print(f"  {name}: {row_sum:.1f} = {balanced_supplies[i]:.1f} ✓")

    print("Столбцы (потребители):")
    for j, name in enumerate(balanced_consumer_names):
        col_sum = plan[:, j].sum()
        print(f"  {name}: {col_sum:.1f} = {balanced_demands[j]:.1f} ✓")

    # Общая стоимость
    total_cost = result.fun
    print(f"\nМинимальная общая стоимость перевозок: {total_cost:.2f}")

    # Проверка через assert
    assert np.allclose(plan.sum(axis=1), balanced_supplies)
    assert np.allclose(plan.sum(axis=0), balanced_demands)
    print("\nВсе проверки пройдены успешно!")

    # Сохраняем план для анализа
    plan_matrix = plan

else:
    print("Решение не найдено!")
    plan_matrix = None

Успех решения: True
Сообщение: Optimization terminated successfully. (HiGHS Status 7: Optimal)

Оптимальный план перевозок:


,Рембаза 1,Рембаза 2,Рембаза 3,Рембаза 4
Склад A,5.0,20.0,0.0,0.0
Склад B,10.0,0.0,0.0,20.0
Склад C,0.0,0.0,18.0,2.0
Фиктивный поставщик,0.0,0.0,0.0,8.0



План перевозок с итогами:


,Рембаза 1,Рембаза 2,Рембаза 3,Рембаза 4,Итого
Склад A,5.0,20.0,0.0,0.0,25.0
Склад B,10.0,0.0,0.0,20.0,30.0
Склад C,0.0,0.0,18.0,2.0,20.0
Фиктивный поставщик,0.0,0.0,0.0,8.0,8.0
Итого,15.0,20.0,18.0,30.0,83.0



Проверка ограничений:
Строки (поставщики):
  Склад A: 25.0 = 25.0 ✓
  Склад B: 30.0 = 30.0 ✓
  Склад C: 20.0 = 20.0 ✓
  Фиктивный поставщик: 8.0 = 8.0 ✓
Столбцы (потребители):
  Рембаза 1: 15.0 = 15.0 ✓
  Рембаза 2: 20.0 = 20.0 ✓
  Рембаза 3: 18.0 = 18.0 ✓
  Рембаза 4: 30.0 = 30.0 ✓

Минимальная общая стоимость перевозок: 459.00

Все проверки пройдены успешно!


In [6]:
# Анализ оптимальных маршрутов
print("\n АНАЛИЗ ОПТИМАЛЬНЫХ МАРШРУТОВ \n")

if plan_matrix is not None:
    # Отделяем реальные и фиктивные маршруты
    real_supplier_count = len(supplier_names)
    dummy_row_index = real_supplier_count
    real_consumer_count = len(consumer_names)

    print(f"Количество реальных поставщиков: {real_supplier_count}")
    print(f"Количество реальных потребителей: {real_consumer_count}")
    print(f"Размер матрицы плана: {plan_matrix.shape}")

    # Проверяем наличие фиктивного поставщика
    has_dummy_supplier = len(balanced_supplier_names) > real_supplier_count
    print(f"Есть фиктивный поставщик: {has_dummy_supplier}")

    print("\nАктивные реальные маршруты (ненулевые перевозки):")
    active_routes = []
    for i, supplier in enumerate(balanced_supplier_names):
        if i >= real_supplier_count:  # пропускаем фиктивного поставщика
            continue
        for j, consumer in enumerate(balanced_consumer_names):
            if j >= real_consumer_count:  # пропускаем фиктивного потребителя
                continue
            amount = plan_matrix[i, j]
            if amount > 0.01:
                cost = costs[i, j]
                total = amount * cost
                active_routes.append({
                    'Маршрут': f"{supplier} → {consumer}",
                    'Объем': f"{amount:.1f}",
                    'Стоимость за ед.': f"{cost:.0f}",
                    'Общая стоимость': f"{total:.1f}"
                })

    if active_routes:
        active_df = pd.DataFrame(active_routes)
        display(active_df)
    else:
        print("Нет активных маршрутов")

    # Анализ фиктивного поставщика (неудовлетворенный спрос)
    if has_dummy_supplier:
        print("\nНЕУДОВЛЕТВОРЕННЫЙ СПРОС (поставки от фиктивного поставщика):")
        dummy_routes = []
        dummy_amounts = []
        for j, consumer in enumerate(balanced_consumer_names):
            if j >= real_consumer_count:
                continue
            amount = plan_matrix[dummy_row_index, j]
            dummy_amounts.append(amount)
            if amount > 0.01:
                dummy_routes.append({
                    'Потребитель': consumer,
                    'Недостающий объем': f"{amount:.1f}",
                    'Причина': 'Недостаток запасов'
                })

        if dummy_routes:
            dummy_df = pd.DataFrame(dummy_routes)
            display(dummy_df)

            total_deficit = sum(dummy_amounts)
            print(f"\nОбщий дефицит (неудовлетворенный спрос): {total_deficit:.1f} единиц")
            print("Это означает, что ремонтные базы не получат необходимые запчасти в полном объеме.")

            # Подробная информация по каждому потребителю
            print("\nДетальный анализ дефицита:")
            for j, consumer in enumerate(balanced_consumer_names):
                if j >= real_consumer_count:
                    continue
                amount = plan_matrix[dummy_row_index, j]
                if amount > 0.01:
                    total_demand = balanced_demands[j]
                    satisfied = plan_matrix[:real_supplier_count, j].sum()
                    pct_satisfied = (satisfied / total_demand) * 100 if total_demand > 0 else 0
                    print(f"  • {consumer}: получено {satisfied:.1f} из {total_demand:.1f} ({pct_satisfied:.1f}%), дефицит {amount:.1f}")
        else:
            print("Нет дефицита (все запросы удовлетворены)")
    else:
        print("\nФиктивный поставщик не добавлен (задача закрыта)")
        dummy_amounts = []

    # Общая стоимость по реальным маршрутам
    if active_routes:
        real_total = sum(float(r['Общая стоимость']) for r in active_routes)
        print(f"\nОбщая стоимость реальных перевозок: {real_total:.1f}")
        print(f"Общая стоимость перевозок (включая фиктивные): {result.fun:.1f}")
        print(f"Совпадение: {'✓' if abs(real_total - result.fun) < 0.01 else '✗'}")
    else:
        print("\nНет активных маршрутов для расчета стоимости")

    print("\n ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТА ")
    if has_dummy_supplier and dummy_amounts:
        total_deficit = sum(dummy_amounts)
        print(f"Общий дефицит запчастей составляет {total_deficit:.1f} единиц")
        print("Это означает, что не все ремонтные базы получат необходимые запчасти.")
        print("Для удовлетворения всего спроса необходимо дополнительно поставить {:.0f} единиц.".format(total_deficit))

    print("\nНаиболее загруженные реальные маршруты (по объему):")
    if active_routes:
        sorted_routes = sorted(active_routes, key=lambda x: float(x['Объем']), reverse=True)
        for route in sorted_routes[:3]:
            print(f"  • {route['Маршрут']}: {route['Объем']} единиц, стоимость {route['Общая стоимость']}")

    if has_dummy_supplier and dummy_routes:
        print("\nПотребители с наибольшим дефицитом:")
        sorted_deficit = sorted(dummy_routes, key=lambda x: float(x['Недостающий объем']), reverse=True)
        for item in sorted_deficit[:2]:
            print(f"  • {item['Потребитель']}: не хватает {item['Недостающий объем']} единиц")

    print("\nИспользование запасов складов:")
    for i, supplier in enumerate(balanced_supplier_names):
        if i >= real_supplier_count:
            continue
        used = plan_matrix[i, :real_consumer_count].sum()
        pct_used = (used / balanced_supplies[i]) * 100 if balanced_supplies[i] > 0 else 0
        print(f"  • {supplier}: использовано {used:.1f} из {balanced_supplies[i]:.1f} ({pct_used:.1f}%)")

    print("\nУдовлетворение спроса ремонтных баз:")
    for j, consumer in enumerate(balanced_consumer_names):
        if j >= real_consumer_count:
            continue
        received = plan_matrix[:real_supplier_count, j].sum()
        pct_received = (received / balanced_demands[j]) * 100 if balanced_demands[j] > 0 else 0
        status = "✓ ПОЛНОСТЬЮ" if pct_received >= 99.9 else f"⚠️ {pct_received:.1f}%"
        print(f"  • {consumer}: получено {received:.1f} из {balanced_demands[j]:.1f} ({status})")

else:
    print("Нет решения для анализа!")


 АНАЛИЗ ОПТИМАЛЬНЫХ МАРШРУТОВ 

Количество реальных поставщиков: 3
Количество реальных потребителей: 4
Размер матрицы плана: (4, 4)
Есть фиктивный поставщик: True

Активные реальные маршруты (ненулевые перевозки):


,Маршрут,Объем,Стоимость за ед.,Общая стоимость
0,Склад A → Рембаза 1,5.0,7,35.0
1,Склад A → Рембаза 2,20.0,5,100.0
2,Склад B → Рембаза 1,10.0,6,60.0
3,Склад B → Рембаза 4,20.0,8,160.0
4,Склад C → Рембаза 3,18.0,5,90.0
5,Склад C → Рембаза 4,2.0,7,14.0



НЕУДОВЛЕТВОРЕННЫЙ СПРОС (поставки от фиктивного поставщика):


,Потребитель,Недостающий объем,Причина
0,Рембаза 4,8.0,Недостаток запасов



Общий дефицит (неудовлетворенный спрос): 8.0 единиц
Это означает, что ремонтные базы не получат необходимые запчасти в полном объеме.

Детальный анализ дефицита:
  • Рембаза 4: получено 22.0 из 30.0 (73.3%), дефицит 8.0

Общая стоимость реальных перевозок: 459.0
Общая стоимость перевозок (включая фиктивные): 459.0
Совпадение: ✓

 ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТА 
Общий дефицит запчастей составляет 8.0 единиц
Это означает, что не все ремонтные базы получат необходимые запчасти.
Для удовлетворения всего спроса необходимо дополнительно поставить 8 единиц.

Наиболее загруженные реальные маршруты (по объему):
  • Склад A → Рембаза 2: 20.0 единиц, стоимость 100.0
  • Склад B → Рембаза 4: 20.0 единиц, стоимость 160.0
  • Склад C → Рембаза 3: 18.0 единиц, стоимость 90.0

Потребители с наибольшим дефицитом:
  • Рембаза 4: не хватает 8.0 единиц

Использование запасов складов:
  • Склад A: использовано 25.0 из 25.0 (100.0%)
  • Склад B: использовано 30.0 из 30.0 (100.0%)
  • Склад C: использовано 20.0 и

In [7]:
# Дополнительный анализ для отчета
print("\n СВОДНАЯ ИНФОРМАЦИЯ \n")

print("Исходные данные:")
print(f"  Общий запас: {supplies.sum():.0f}")
print(f"  Общий спрос: {demands.sum():.0f}")
print(f"  Дефицит: {demands.sum() - supplies.sum():.0f}")

if plan_matrix is not None:
    real_supplier_count = len(supplier_names)
    real_consumer_count = len(consumer_names)

    print("\nОптимальный план:")
    # Создаем сводную таблицу
    summary_data = []
    for i, supplier in enumerate(supplier_names):
        row_data = {'Склад': supplier}
        for j, consumer in enumerate(consumer_names):
            row_data[consumer] = plan_matrix[i, j]
        # Добавляем фиктивного поставщика, если есть
        if len(balanced_supplier_names) > real_supplier_count:
            row_data['Дефицит'] = plan_matrix[real_supplier_count, j]
        row_data['Всего'] = plan_matrix[i, :real_consumer_count].sum()
        summary_data.append(row_data)

    summary_df = pd.DataFrame(summary_data)
    display(summary_df)

    # Экономическая эффективность
    print("\nЭкономическая эффективность:")
    if active_routes:
        total_cost = result.fun
        total_volume = supplies.sum()
        avg_cost = total_cost / total_volume if total_volume > 0 else 0
        print(f"  Средняя стоимость перевозки единицы: {avg_cost:.2f}")

        # Находим самый дешевый маршрут
        cheapest = min(active_routes, key=lambda x: float(x['Стоимость за ед.']))
        print(f"  Самый дешевый маршрут: {cheapest['Маршрут']} (стоимость {cheapest['Стоимость за ед.']} за единицу)")

        # Находим самый дорогой маршрут
        expensive = max(active_routes, key=lambda x: float(x['Стоимость за ед.']))
        print(f"  Самый дорогой маршрут: {expensive['Маршрут']} (стоимость {expensive['Стоимость за ед.']} за единицу)")

    print("\nРекомендации:")
    if len(balanced_supplier_names) > real_supplier_count:
        total_deficit = sum(plan_matrix[real_supplier_count, :real_consumer_count])
        if total_deficit > 0:
            print(f"  • Необходимо дополнительно закупить {total_deficit:.0f} единиц запчастей")
            print("  • Приоритетные потребители с наибольшим дефицитом:")
            for j, consumer in enumerate(consumer_names):
                deficit = plan_matrix[real_supplier_count, j]
                if deficit > 0.01:
                    print(f"    - {consumer}: {deficit:.0f} единиц")
    else:
        print("  • Все потребности полностью удовлетворены")
        print("  • Запасы распределены оптимально")


 СВОДНАЯ ИНФОРМАЦИЯ 

Исходные данные:
  Общий запас: 75
  Общий спрос: 83
  Дефицит: 8

Оптимальный план:


,Склад,Рембаза 1,Рембаза 2,Рембаза 3,Рембаза 4,Дефицит,Всего
0,Склад A,5.0,20.0,0.0,0.0,8.0,25.0
1,Склад B,10.0,0.0,0.0,20.0,8.0,30.0
2,Склад C,0.0,0.0,18.0,2.0,8.0,20.0



Экономическая эффективность:
  Средняя стоимость перевозки единицы: 6.12
  Самый дешевый маршрут: Склад A → Рембаза 2 (стоимость 5 за единицу)
  Самый дорогой маршрут: Склад B → Рембаза 4 (стоимость 8 за единицу)

Рекомендации:
  • Необходимо дополнительно закупить 8 единиц запчастей
  • Приоритетные потребители с наибольшим дефицитом:
    - Рембаза 4: 8 единиц


## 5. Что должно быть в отчёте

1. Таблица исходных данных и проверка баланса.
2. Полная математическая постановка через $x_{ij}$.
3. Пояснение, нужен ли фиктивный поставщик или фиктивный потребитель.
4. Векторно-матричная запись `c`, `A_eq`, `b_eq`, `bounds`.
5. Проверка через `linprog`.
6. Таблица оптимального плана перевозок.
7. Проверка сумм по строкам и столбцам.
8. Содержательная интерпретация ненулевых маршрутов.

## 6. Контрольный чек-лист

- [ ] Я сам проверил баланс до запуска solver-а.
- [ ] Я осмысленно собрал `A_eq` и `b_eq` через индекс `route_index`.
- [ ] Я проверил `bounds` и неотрицательность всех $x_{ij}$.
- [ ] Я отличаю реальные маршруты от фиктивного узла.
- [ ] Я объяснил полученный план простыми словами.
